# Reproduce the main benchmark plots

This notebook reproduces Figures 2–4 of [HomID, arXiv v2](https://arxiv.org/abs/2510.01335v2). It uses `benchmark_data.npz`. It does not run estimators or load the original point clouds.

The bundle contains only the values used by these figures. It retains the three seeds before the final average. All floating-point arrays retain their original `float64` precision. NumPy applies lossless ZIP compression. See `benchmark_manifest.json` for source files, dates, hashes, array shapes, and aggregation rules.

The plotting cells are copied from the author's latest plotting notebooks. File paths now use this release folder. Figure 3 has the correct axis labels: distortion strength on x and signed relative error on y. The source notebook had these labels interchanged. Numeric curves are unchanged.

Run all cells from the repository root or from `plots/`. Figures are saved under `plots/generated/`. The final section can rebuild the compact bundle from a local copy of the original results. It is disabled by default.


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

ROOT = Path.cwd() if (Path.cwd() / "data" / "benchmark_data.npz").is_file() else Path.cwd().parent
DATA_DIR = ROOT / "data"
if not (DATA_DIR / "benchmark_data.npz").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or plots/.")
OUTPUT_DIR = ROOT / "plots" / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with np.load(DATA_DIR / "benchmark_data.npz", allow_pickle=False) as archive:
    values = {name: archive[name] for name in archive.files}
manifest = json.loads((DATA_DIR / "benchmark_manifest.json").read_text())
assert hashlib.sha256((DATA_DIR / "benchmark_data.npz").read_bytes()).hexdigest() == manifest["bundle_sha256"]
print(f"Loaded {len(values)} arrays from {manifest['bundle']} ({manifest['bundle_bytes']:,} bytes).")


## Figure 2. Compare HomID with matched spheres

For each family and method, the heatmap shows the ratio of the two mean absolute relative errors, minus one. Colors are clipped to `[-1, 1]`. Cell values retain the full score. The text after ± is the sum of the two across-seed population standard deviations. This matches the paper plotting code. It is not an error propagated through the ratio. One original HomID entry is NaN (Flag, TwoNN, seed 42). The average omits that entry, as in the original notebook. The missing value is retained in the bundle.

In [ ]:
IDE_NAMES = values["fig2_methods"].tolist()
IDE_LABELS = ["lPCA", "MLE", "CorrInt", "TwoNN", "ABID", "DANCo"]
MANIFOLDS = values["fig2_families"].tolist()
MANIFOLD_LABELS = ["Gr (Proj)", "Gr (Vec)", "Flag (Vec)", "St (Matrix)", "St (Vec)"]
seed_errors = values["fig2_seed_mean_abs_error"]
seed_means = np.nanmean(seed_errors, axis=2)
seed_sd = np.nanstd(seed_errors, axis=2)
mean_errors = np.empty((len(IDE_NAMES), len(MANIFOLDS)), dtype=object)
seed_spreads = np.empty_like(mean_errors)
for j in range(len(MANIFOLDS)):
    for i in range(len(IDE_NAMES)):
        mean_errors[i, j] = tuple(seed_means[j, i])
        seed_spreads[i, j] = (tuple(seed_sd[j, i]), (np.nan, np.nan))
Mean_Abs_Errors = {"sphere": mean_errors}
Stdev_Seeds = {"sphere": seed_spreads}

comparison_labels = ['sphere', 'gaussian', 'affine', 'mbeta']

titles = {
    'sphere': r'$\langle|\delta|\rangle_{M}/\langle|\delta|\rangle_{S} - 1$',
    'gaussian': r'$\langle|\delta|\rangle_{M}/\langle|\delta|\rangle_{N} - 1$',
    'affine': r'$\langle|\delta|\rangle_{M}/\langle|\delta|\rangle_{L} - 1$',
    'mbeta': r'$\langle|\delta|\rangle_{M}/\langle|\delta|\rangle_{M_\beta} - 1$'
}

CLIP_VAL = 1.0

for label in ['sphere']:
    MMM = Mean_Abs_Errors[label]
    MMM3 = Stdev_Seeds[label]

    valid_manifold_indices = []
    valid_manifold_labels = []
    for j, mn in enumerate(MANIFOLDS):
        if 'pauli' in mn:
            continue
        valid_manifold_indices.append(j)
        valid_manifold_labels.append(MANIFOLD_LABELS[j])

    diff_mat = np.full((len(valid_manifold_indices), len(IDE_NAMES)), np.nan, dtype=float)
    text_mat = [[None for _ in range(len(IDE_NAMES))] for _ in range(len(valid_manifold_indices))]

    for row_idx, j in enumerate(valid_manifold_indices):
        for col_idx in range(len(IDE_NAMES)):
            val = MMM[col_idx, j]
            if val is not None and not np.isnan(val[0]) and not np.isnan(val[1]) and val[1] != 0:
                diff_mat[row_idx, col_idx] = (val[0] - val[1]) / val[1]
            else:
                diff_mat[row_idx, col_idx] = np.nan

            if not np.isnan(diff_mat[row_idx, col_idx]):
                stdev_combined = MMM3[col_idx, j][0][0] + MMM3[col_idx, j][0][1]
                text_mat[row_idx][col_idx] = f'{diff_mat[row_idx, col_idx]:.4f}\n±{stdev_combined:.2e}'
            else:
                text_mat[row_idx][col_idx] = 'N/A'

    fig, ax = plt.subplots(figsize=(14, 6))
    cax = ax.imshow(diff_mat, cmap='coolwarm', vmin=-CLIP_VAL, vmax=CLIP_VAL)

    ax.set_xticks(np.arange(len(IDE_NAMES)))
    ax.set_xticklabels(IDE_LABELS, fontsize=12)
    ax.set_yticks(np.arange(len(valid_manifold_indices)))
    ax.set_yticklabels(valid_manifold_labels, fontsize=12)
    ax.set_title(titles[label], fontsize=18)

    for i in range(len(valid_manifold_indices)):
        for j in range(len(IDE_NAMES)):
            rect = Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor='black', linewidth=1)
            ax.add_patch(rect)

    for i in range(len(valid_manifold_indices)):
        for j in range(len(IDE_NAMES)):
            ax.text(j, i, text_mat[i][j], ha='center', va='center', fontsize=11)

    fig.colorbar(cax, ax=ax, shrink=0.5)
    fn = OUTPUT_DIR / 'new_compare_rel_sphere_only.pdf'
    fig.savefig(fn, format='pdf', bbox_inches='tight')
    plt.tight_layout()
    plt.show()


## Figure 3. Apply diagonal distortion

The curves show mean signed relative error. The shaded bands show the average within-seed standard deviation of local estimates. They do not show uncertainty in the mean. The three seeds are retained in the bundle. The original epsilon grid order is preserved.

In [ ]:
IDE_NAMES = values["fig3_methods"].tolist()
IDE_LABELS = {"lpca_maxgap": "lPCA", "mle": "MLE", "corrint": "CorrInt", "danco": "DANCo", "twonn": "TwoNN", "abid": "ABID"}
MANIFOLD_LABELS = {"steifel_proj": "St (Matrix)", "steifel_vec1": "St (Vec)", "flag_vec": "Flag (Vec)"}
INFO_DICT = {"sq_eps": values["fig3_epsilon"]}
for j, mn in enumerate(values["fig3_families"]):
    INFO_DICT[mn] = {"di,da": tuple(values["fig3_dimensions"][j])}
    for i, ide in enumerate(IDE_NAMES):
        INFO_DICT[mn][ide] = {
            "mean": np.nanmean(values["fig3_seed_mean_error"][j, i], axis=-1),
            "std": np.nanmean(values["fig3_seed_local_std"][j, i], axis=-1),
        }

fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=False)
axs = axs.flatten()

MANIFOLD_TO_PLOT = [# 'gman_proj', 'gman_vec', 
    'steifel_proj',
             'steifel_vec1', 'flag_vec',
               ]

label_fontsize = 18
tick_fontsize = 16
legend_fontsize = 18
marker = '.'

row_labels = ['Common Y Label Row 1', 'Common Y Label Row 2']
col_labels = ['Common X Label Col 1', 'Col 2', 'Col 3']

ide_colors = {ide: color for ide, color in zip(IDE_NAMES, plt.cm.tab10.colors)}

for idx, mn in enumerate(MANIFOLD_TO_PLOT):
    ax = axs[idx]

    for ide in IDE_NAMES:
        mean_vals = INFO_DICT[mn][ide]['mean']
        std_vals = INFO_DICT[mn][ide]['std']
        
        color = ide_colors[ide]
        
        ax.plot(INFO_DICT['sq_eps'], mean_vals, label=IDE_LABELS[ide], color=color, marker=marker)
        ax.fill_between(INFO_DICT['sq_eps'], mean_vals - std_vals, mean_vals + std_vals,
                        alpha=0.3, color=color)

    di, da = INFO_DICT[mn]['di,da']
    ax.axhline(0, color='black', linestyle='--', linewidth=3)
    ax.axhline(da/di - 1, color='black', linestyle='-.', linewidth=3)

    ax.tick_params(axis='both', labelsize=tick_fontsize)
    ax.set_title(MANIFOLD_LABELS[mn], fontsize=label_fontsize)
    ax.set_xlabel('$\epsilon$', fontsize=label_fontsize)

    ax.grid(True)

axs[0].set_ylabel('$\delta$', fontsize=label_fontsize)
    

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.00, 0.5), 
           fontsize=legend_fontsize)

plt.tight_layout(rect=[0, 0.0, 1.0, 1.2])  # Shift content left to make space on the right
plt.show()

SAVE_FIGNAME = OUTPUT_DIR / 'score_vs_epsilon_sub.pdf'
fig.savefig(SAVE_FIGNAME, format='pdf', bbox_inches='tight')


## Figure 4. Add trace-normalized noise

The x axis is the original noise variance parameter, `sigma²`. The three line styles select the three covariance types. The bands use the average within-seed standard deviation of local estimates. Their opacity matches the source plotting cell.

In [ ]:
IDE_NAMES = values["fig4_methods"].tolist()
IDE_LABELS = ["lPCA", "MLE", "TwoNN", "CorrInt", "DANCo", "ABID"]
noise_levels = values["fig4_sigma_squared"]
Noise_Dict = {"int_dims": {}}
for nt in values["fig4_noise_types"]:
    Noise_Dict[nt] = {}
    Noise_Dict[f"{nt}_std"] = {}
for j, mn in enumerate(values["fig4_families"]):
    for i, ide in enumerate(IDE_NAMES):
        key = f"{ide}_{mn}"
        di, da = values["fig4_dimensions"][j]
        Noise_Dict["int_dims"][key] = (da, di)
        for t, nt in enumerate(values["fig4_noise_types"]):
            Noise_Dict[nt][key] = np.nanmean(values["fig4_seed_mean_error"][j, i, t], axis=-1)
            Noise_Dict[f"{nt}_std"][key] = np.nanmean(values["fig4_seed_local_std"][j, i, t], axis=-1)

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.lines as mlines

noise_types = ['isoGauss', 'anisoGauss', 'uncorrGauss']

MANIFOLD_LABELS = {'lin_nsp':'Affine Nullspace',
                   'gman_proj':'Gr (Proj)',
                   'gman_vec': 'Gr (Vec)',
                   'steifel_proj' : 'St (Matrix)',
                   'steifel_vec1' : 'St (Vec)',
                   'flag_vec': 'Flag (Vec)',
                   'sun_pauli' : 'Pauli'}

MANIFOLD_TO_PLOT = [# 'lin_nsp', 
                  'gman_vec', 
                  'steifel_vec1', 
                  'flag_vec', 
                  ]

ide_colors = {}

for c in range(len(IDE_NAMES)):
    ide_colors[IDE_LABELS[c]] = f'C{c}'

noise_linestyles = {'isoGauss': '-', 'uncorrGauss': '--', 'anisoGauss': ':'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=False)
axes = axes.flatten() # Flatten the 2x3 array for easy iteration

for i, mn in enumerate(MANIFOLD_TO_PLOT):
    ax = axes[i]
    
    for ide, ide_label in zip(IDE_NAMES, IDE_LABELS):
        color = ide_colors[ide_label]
        
        for nt in noise_types:
            key = f'{ide}_{mn}'
            linestyle = noise_linestyles[nt]
            
            y_vals = np.array(Noise_Dict[nt][key])
            std_vals = np.array(Noise_Dict[f'{nt}_std'][key])
            
            ax.plot(noise_levels, y_vals,
                    color=color,
                    linestyle=linestyle,
                    marker='.',
                    markersize=5,
                    label=f'{ide} - {nt}')
            
            ax.fill_between(noise_levels, y_vals - std_vals, y_vals + std_vals,
                            color=color,
                            alpha=0.01)

    ax.set_title(MANIFOLD_LABELS[mn], fontsize=16)
    ax.set_xscale('log')
    ax.grid(True)
    ad,id = Noise_Dict['int_dims'][key]
    ax.axhline(ad/id-1, ls = '-.', color ='black')
    ax.axhline(0.0, ls = '--', color ='black')
    ax.tick_params(axis='both',labelsize=16)


fig.supxlabel(r'$\sigma^2$', fontsize=18,)
fig.supylabel(r'$\delta$', fontsize=18,)


color_handles = [mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                               markersize=10, label=ide) for ide, c in ide_colors.items()]

linestyle_handles = [mlines.Line2D([], [], color='gray', linestyle=ls,
                                   label=nt) for nt, ls in noise_linestyles.items()]

all_handles = color_handles + linestyle_handles
fig.legend(handles=all_handles, 
           loc='center right', 
           bbox_to_anchor=(1.0, 0.5), 
           fontsize=14, 
           borderaxespad=0.1,
           title="Legend",
           title_fontsize=16)

plt.tight_layout(rect=[0.02, 0.0, 0.89, 0.95]) # rect=[left, bottom, right, top]

pn = OUTPUT_DIR / 'noise_tre_scaling_rel_sub.pdf'
fig.savefig(pn, format ='pdf')
plt.show()


## Check compression

This cell writes a compressed copy under `generated/`. It checks every array, including NaN values. The release archive remains unchanged.

The seed coordinate arrays identify the retained seed axes. They support comparisons across seeds and document which original runs contribute to each plot.

In [ ]:
# Verify a lossless compression round trip without changing the release archive.
roundtrip_path = OUTPUT_DIR / "benchmark_data_roundtrip.npz"
np.savez_compressed(roundtrip_path, **values)
with np.load(roundtrip_path, allow_pickle=False) as restored:
    assert set(restored.files) == set(values)
    for name, array in values.items():
        np.testing.assert_array_equal(restored[name], array)
print(f"Lossless round trip passed: {roundtrip_path.stat().st_size:,} bytes.")


## Optional: rebuild the summary from original results

This section contains the extraction and compression recipe. It reads the local original results only when `REBUILD_FROM_ORIGINALS` is `True`. All writes go to this release folder. It preserves the original source notebooks and data.

The source notebooks select result files by experiment type and fallback order. The manifest records the selected files, modification times in UTC, and SHA-256 hashes. This is more precise than selecting files by modification time alone. The selected figure assets match the paper: Figures 2 and 4 are byte-identical, and Figure 3 has the same decoded PDF drawing stream.

For Figure 2, the recipe executes the author's existing read-only score functions. For Figures 3 and 4, it retains the exact per-seed statistics used by the plotting code. Large arrays such as point clouds, neighbor data, and PCA diagnostics are discarded after reading. No estimator is rerun.

In [ ]:
def export_benchmark_summary(original_root, output_dir):
    from pathlib import Path
    from datetime import datetime, timezone
    import builtins, contextlib, hashlib, io, json, math, pickle, warnings
    import numpy as np
    ROOT = Path(original_root).expanduser().resolve()
    OUT = Path(output_dir).expanduser().resolve()
    if OUT.is_relative_to(ROOT):
        raise ValueError("output_dir must be outside original_root. Original files must remain unchanged.")
    OUT.mkdir(exist_ok=True)
    notebook = json.loads((ROOT/'plotting_2.ipynb').read_text())
    review = json.loads((ROOT/'review_plots.ipynb').read_text())
    used = {2:set(),3:set(),4:set()}
    figure = 2

    def source_open(path, mode='rb', *a, **kw):
        if mode not in ('r','rb'):
            raise ValueError('Original source data is read-only.')
        p = ROOT / path
        fp = builtins.open(p, mode, *a, **kw)
        used[figure].add(p)
        return fp

    scope = {'np':np, 'math':math, 'pickle':pickle, 'open':source_open}
    exec(''.join(notebook['cells'][28]['source']), scope)
    exec(''.join(notebook['cells'][32]['source']), scope)
    # These dimensions are recorded in the original cell 29 output.
    family_dims = {'gman_proj':[(2,9),(4,16),(3,16)],
                   'gman_vec':[(2,3),(4,6),(3,4)],
                   'flag_vec':[(4,36),(12,225)],
                   'st_proj':[(5,8),(3,12)],
                   'st_vec':[(5,10),(3,13)]}
    methods2 = ['lpca_maxgap','mle','corrint','twonn','abid','danco']
    families2 = list(family_dims)
    seed2 = [42,84,126]
    err2 = np.empty((len(families2),len(methods2),len(seed2),2))
    std2 = np.empty_like(err2)
    with contextlib.redirect_stdout(io.StringIO()), warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        for j,mn in enumerate(families2):
            for i,ide in enumerate(methods2):
                for z,seed in enumerate(seed2):
                    mf, ms, _ = scope['score_ide_bm'](mn,ide,family_dims,SEED=seed)
                    sp, ss, _ = scope['isometry_score_ide_bm'](mn,ide,family_dims,SEED=seed)
                    err2[j,i,z] = [mf['manifold'],sp['sphere']]
                    std2[j,i,z] = [ms['manifold'],ss['sphere']]
    arrays = {'fig2_methods': np.array(methods2), 'fig2_families':np.array(families2),
              'fig2_seeds':np.array(seed2), 'fig2_seed_mean_abs_error':err2}
    # Mean local spread is deliberately omitted: Figure 2 only uses mean errors and their seed SD.
    print('fig2 errors',err2.shape, 'first column',err2[0].mean(axis=1))

    figure = 3
    # Copy the exact epsilon grid and parameter choices from review_plots.ipynb cell 6.
    review_scope={'np':np,'copy':__import__('copy')}
    exec(''.join(review['cells'][6]['source']), review_scope)
    methods3 = review_scope['IDE_NAMES']
    families3 = ['steifel_proj','steifel_vec1','flag_vec']
    eps = review_scope['SQ_EPS_VALS']
    means3 = np.empty((3,6,len(eps),3)); spread3=np.empty_like(means3)
    dims3 = np.array([[145,200],[26,151],[9,400]]) # (intrinsic, ambient)
    for j,mn in enumerate(families3):
        mp=review_scope['MANIFOLD_PARAMS'][mn]+(42,)
        for i,ide in enumerate(methods3):
            for z in range(3):
                for q,sq in enumerate(eps):
                    params=review_scope['IDE_PARAMS'][ide]+(z,5000,0.0,float(sq))
                    p=Path('quiiest/new_codes/data/squeeze_no_curve')/f'_{ide}_p_{params}_sq_{mn}_mp_{mp}.pkl'
                    with source_open(p,'rb') as f: obj=pickle.load(f)
                    x=np.asarray(obj['lides'])
                    means3[j,i,q,z]=np.nanmean(x)/dims3[j,0]-1
                    spread3[j,i,q,z]=np.nanstd(x)/dims3[j,0]
                    del obj,x
    arrays.update(fig3_methods=np.array(methods3),fig3_families=np.array(families3),
                  fig3_epsilon=eps,fig3_seeds=np.arange(3),fig3_dimensions=dims3,
                  fig3_seed_mean_error=means3,fig3_seed_local_std=spread3)
    print('fig3',means3.shape)

    figure=4
    methods4=['lpca_maxgap','mle_e','twonn_e','corrint','danco','abid']
    families4=['gman_vec','steifel_vec1','flag_vec']
    mp4=[(11,5,42),(9,5,42),(6,3,3,42)]
    dims4=np.array([[30,462],[26,151],[9,400]])
    noise=['isoGauss','anisoGauss','uncorrGauss']
    # This literal grid, including its precision, controls the original result filenames.
    sigmas=np.array([1.00000000e-05,2.06913808e-05,4.28133240e-05,8.85866790e-05,
     1.83298071e-04,3.79269019e-04,7.84759970e-04,1.62377674e-03,
     3.35981829e-03,6.95192796e-03,1.43844989e-02,2.97635144e-02,
     6.15848211e-02,1.27427499e-01,2.63665090e-01,5.45559478e-01,
     1.12883789e+00,2.33572147e+00,4.83293024e+00,1.00000000e+01])
    base_params=[(200,None),(100,0.0),(100,0.1),(10,20),(10,10),(100,)]
    means4=np.empty((3,6,3,len(sigmas),3)); spread4=np.empty_like(means4)
    for j,mn in enumerate(families4):
        for i,ide in enumerate(methods4):
            for t,nt in enumerate(noise):
                for z in range(3):
                    for q,sigma in enumerate(sigmas):
                        params=base_params[i]+(z,4000,float(sigma),nt)
                        p=Path('data/new_noise_data')/f'_{ide}_p_{params}_{mn}_mp_{mp4[j]}.pkl'
                        with source_open(p,'rb') as f: obj=pickle.load(f)
                        x=np.asarray(obj['lides'])
                        # The source notebook uses mean/std, not nanmean/nanstd, within a seed.
                        means4[j,i,t,q,z]=np.mean(x)/dims4[j,0]-1
                        spread4[j,i,t,q,z]=np.std(x)/dims4[j,0]
                        del obj,x
    arrays.update(fig4_methods=np.array(methods4),fig4_families=np.array(families4),
                  fig4_noise_types=np.array(noise),fig4_sigma_squared=sigmas,
                  fig4_seeds=np.arange(3),fig4_dimensions=dims4,
                  fig4_seed_mean_error=means4,fig4_seed_local_std=spread4)
    print('fig4',means4.shape)
    np.savez_compressed(OUT/'benchmark_data.npz',**arrays)

    def metadata(p):
        h=hashlib.sha256()
        with builtins.open(p,'rb') as f:
            while chunk:=f.read(1024*1024):h.update(chunk)
        return {'path':p.relative_to(ROOT).as_posix(),'bytes':p.stat().st_size,
                'mtime_utc':datetime.fromtimestamp(p.stat().st_mtime,timezone.utc).isoformat(),
                'sha256':h.hexdigest()}

    all_sources=sorted(set.union(*used.values()))
    manifest={'paper':'https://arxiv.org/abs/2510.01335v2',
     'source_root_name':'local_ideb',
     'missing_values':[{'figure':2,'family':'flag_vec','method':'twonn','seed':42,'sample':'HomID','treatment':'Original NaN retained. Across-seed nanmean/nanstd omit this entry, as in the original plotting notebook.'}],
     'bundle':'benchmark_data.npz',
     'bundle_sha256':hashlib.sha256((OUT/'benchmark_data.npz').read_bytes()).hexdigest(),
     'compression':'numpy.savez_compressed; original float64; no rounding',
     'scope':'Main-text Figures 2, 3 and 4 only. No point clouds, neighbor arrays, unused diagnostic arrays, or appendix runs.',
     'notebooks':[metadata(ROOT/'plotting_2.ipynb'),metadata(ROOT/'review_plots.ipynb')],
     'figures':{
     '2':{'plot':'new_compare_rel_sphere_only.pdf','notebook':'plotting_2.ipynb','cells':[28,29,32,35,40],
          'retained_fields':'Mean absolute relative error per family, method, and seed for HomID and isometric spheres.',
          'aggregation':'Each seed first averages absolute relative error across all saved fixed_N, fixed_k and fixed_kN sweeps in an instantiation, then averages instantiations equally. Across seeds, take nanmean. Plot HomID mean / sphere mean - 1. Text uncertainty is the sum of the two across-seed population SDs, exactly as in the original plot. It is not a propagated ratio error.',
          'source_selection':'Original score_ide_bm source selection including its fallback behavior; sphere values replaced by isometry_score_ide_bm from compare_review (large, then small, then temp). The exporter executes the original read-only functions. Unused baseline branches are read by those functions but their data is not retained.',
          'sources':sorted(p.relative_to(ROOT).as_posix() for p in used[2])},
     '3':{'plot':'score_vs_epsilon_sub.pdf','notebook':'review_plots.ipynb','cells':[6,7,11],
          'retained_fields':'Per-seed mean signed relative error and within-seed local population SD, epsilon grid, dimensions, seeds, method and family labels.',
          'aggregation':'Within a seed use nanmean(lides)/d_i-1 and nanstd(lides)/d_i. Average each quantity across seeds with nanmean; shade mean +/- mean local SD.',
          'presentation_adjustments':'Axis labels corrected to x=epsilon and y=delta. Original cell 11 interchanged these labels. Numeric curves and shading are unchanged.',
          'sources':sorted(p.relative_to(ROOT).as_posix() for p in used[3])},
     '4':{'plot':'noise_tre_scaling_rel_sub.pdf','notebook':'plotting_2.ipynb','cells':[14,15,19],
          'retained_fields':'Per-seed mean signed relative error and within-seed local population SD, original sigma-squared grid, dimensions, seeds, noise type, method and family labels.',
          'aggregation':'Within a seed use mean(lides)/d_i-1 and std(lides)/d_i. Average each quantity across seeds with nanmean; shade mean +/- mean local SD at alpha=0.01. The input sigma parameter denotes noise variance in this experiment.',
          'sources':sorted(p.relative_to(ROOT).as_posix() for p in used[4])}},
     'arrays':{k:{'shape':list(v.shape),'dtype':str(v.dtype)} for k,v in arrays.items()},
     'sources':[metadata(p) for p in all_sources],
     'source_bytes':sum(p.stat().st_size for p in all_sources),
     'bundle_bytes':(OUT/'benchmark_data.npz').stat().st_size}
    (OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    print('saved',manifest['bundle_bytes'],'bytes from',len(all_sources),'source files',manifest['source_bytes'],'bytes')


In [ ]:
# Set ORIGINAL_ROOT to your local results folder, then enable the rebuild.
REBUILD_FROM_ORIGINALS = False
ORIGINAL_ROOT = None
if REBUILD_FROM_ORIGINALS:
    if ORIGINAL_ROOT is None:
        raise ValueError("Set ORIGINAL_ROOT to the local_ideb results folder first.")
    export_benchmark_summary(ORIGINAL_ROOT, DATA_DIR)
else:
    print("Using the release bundle. Original data was not read.")
